# W8 · Day 1 — Parsing Real Documents & Chunking Strategies

**~75 minutes · in-class demo · Jupyter notebook · Track A**

This week is the **ingestion week** — messier than W7, less abstract than W9.
Day 1 covers the two mechanical foundations: **parsing real document formats**
(PDF, HTML, DOCX) and **choosing a chunking strategy** that respects structure.

Day 2 will cover metadata, PII, and wire everything into a mini-pipeline.

**Corpus:** 3 sample documents bundled in `sample_docs/` — a policies PDF,
a product HTML page, and an onboarding DOCX. These are generated by
`generate_sample_docs.py` (rerun to regenerate; substitute your own if
you want more variety).

**Cost per full run:** ~$0.001 (just a few embedding calls for the semantic
chunking demo).

**Notebook flow:**
- Cell 1: Setup + confirm sample docs exist
- Cells 2-6: Parsing — PyMuPDF, pdfplumber, BeautifulSoup, python-docx, the scanned-PDF trap
- Cell 7: Recap Topic 1
- Cells 8-11: Chunking — fixed, recursive, semantic, structure-aware
- Cell 12: Wrap + Presidio install homework for Day 2

**Before Day 2 you must:** `pip install presidio-analyzer presidio-anonymizer` and
`python -m spacy download en_core_web_sm`. See end-of-notebook reminder.

---

## Cell 1 — Setup

Import parsers and confirm the sample documents are present. If any are
missing, run `python demos/generate_sample_docs.py` first.

In [ ]:
from pathlib import Path

SAMPLE_DIR = Path.cwd() / "sample_docs"

SAMPLE_PDF  = SAMPLE_DIR / "company_policies.pdf"
SAMPLE_HTML = SAMPLE_DIR / "product_page.html"
SAMPLE_DOCX = SAMPLE_DIR / "onboarding.docx"

# Hard-fail if any are missing — the notebook depends on them
for path in [SAMPLE_PDF, SAMPLE_HTML, SAMPLE_DOCX]:
    assert path.exists(), (
        f"Missing {path.name}. Run: python demos/generate_sample_docs.py"
    )

print("Sample documents ready:")
for path in [SAMPLE_PDF, SAMPLE_HTML, SAMPLE_DOCX]:
    print(f"  {path.name:30s}  {path.stat().st_size:>6d} bytes")

---

## Cell 2 — PyMuPDF on the sample PDF

**PyMuPDF (imported as `fitz` or `pymupdf`)** is the primary PDF library.
Fast, good text extraction, handles most PDFs.

Extract text page-by-page and observe what comes back.

In [ ]:
import pymupdf  # aka fitz

doc = pymupdf.open(SAMPLE_PDF)
print(f"PDF: {SAMPLE_PDF.name}")
print(f"Pages: {len(doc)}\n")

for page_num, page in enumerate(doc, 1):
    text = page.get_text()
    print(f"══ Page {page_num} ({len(text)} chars) ══")
    # Show first 250 chars of each page
    preview = text[:250].replace('\n', ' | ')
    print(f"{preview}...\n")

doc.close()

**Discussion moment:**
- Notice PyMuPDF preserves paragraph breaks (as `\n\n`) — this helps chunking later
- Look at page 1 — the table appears as jumbled columns. **PyMuPDF struggles with tables.**
- For pure paragraph text, PyMuPDF is fine. For tables, we need something better — next cell.

---

## Cell 3 — pdfplumber on the same PDF

**pdfplumber** is slower than PyMuPDF but much better at tables. Use it when
your corpus has structured tabular data.

Same file, watch the difference on the leave-approval table.

In [ ]:
import pdfplumber

with pdfplumber.open(SAMPLE_PDF) as pdf:
    first_page = pdf.pages[0]
    
    # Text extraction — similar to PyMuPDF for prose
    text = first_page.extract_text()
    print("Text extraction (same as PyMuPDF for prose):")
    print(text[:300], "...\n")
    
    # Table extraction — pdfplumber's superpower
    tables = first_page.extract_tables()
    print(f"Tables detected on page 1: {len(tables)}")
    for i, table in enumerate(tables, 1):
        print(f"\n══ Table {i} ══")
        for row in table:
            print(f"  {' | '.join(str(c) for c in row)}")

**Discussion moment:**
- pdfplumber returns the table as a list-of-lists — proper structure preserved
- For RAG, this matters: a chunk of `'Annual | 20 | Line manager'` is useful; a jumbled
  string like `'Annual 20 Line manager'` might work but loses column semantics
- **Rule of thumb:** try PyMuPDF first (fast). If your corpus has tables, add pdfplumber
  as a fallback and merge the outputs.

---

## Cell 4 — BeautifulSoup on the HTML page

**BeautifulSoup** parses HTML into a navigable tree. For RAG, the important
operation is **stripping non-content** (nav, footer, script, style) before
extracting text.

Watch what happens if you DON'T strip — the retrieval-relevant content gets
buried in menu items and copyright notices.

In [ ]:
from bs4 import BeautifulSoup

html = SAMPLE_HTML.read_text()

# Naive: get all text, no stripping
soup_naive = BeautifulSoup(html, "html.parser")
naive_text = soup_naive.get_text(separator="\n", strip=True)
print("══ NAIVE extraction (nav + footer + scripts included) ══")
print(naive_text[:400])
print(f"\n... {len(naive_text)} chars total\n")

In [ ]:
# Better: strip non-content tags first
soup_clean = BeautifulSoup(html, "html.parser")
for tag in soup_clean(["nav", "footer", "script", "style"]):
    tag.decompose()

# Get <main> if present, else body
main = soup_clean.find("main") or soup_clean.find("body") or soup_clean
clean_text = main.get_text(separator="\n", strip=True)

print("══ CLEAN extraction (nav + footer + scripts stripped) ══")
print(clean_text[:400])
print(f"\n... {len(clean_text)} chars total")
print(f"\nRemoved: {len(naive_text) - len(clean_text)} chars of noise")

**Discussion:** the naive extraction leaks 'Home | Products | Pricing | Docs'
into the top of your chunk. Retrieval then matches that noise against user
queries about 'home page' or 'pricing' — retrieving your product page for
questions it shouldn't answer.

**Rule of thumb for HTML:** always strip `<nav>`, `<footer>`, `<script>`,
`<style>` before extracting text. Add `<aside>` if your source has sidebars.

---

## Cell 5 — python-docx on the DOCX

**python-docx** parses DOCX files (Word documents). Each paragraph has a
`style` — 'Title', 'Heading 1', 'Heading 2', 'Normal', etc. These styles are
**gold** for structure-aware chunking (Cell 11).

In [ ]:
from docx import Document

doc = Document(str(SAMPLE_DOCX))

print(f"DOCX: {SAMPLE_DOCX.name}")
print(f"Paragraphs: {len(doc.paragraphs)}\n")

for i, para in enumerate(doc.paragraphs):
    if para.text.strip():
        style = para.style.name
        preview = para.text[:70]
        print(f"  [{i:2d}] {style:15s}  {preview}...")

**Notice the `style` column.** Heading 1 / Heading 2 tell us where sections
start. That structure is exactly what we'll use in Cell 11 for structure-aware
chunking.

PyMuPDF and BeautifulSoup don't give you clean structure like this by default —
DOCX is the friendliest format for ingestion.

---

## Cell 6 — The scanned PDF trap

**Some PDFs contain no text — only images of pages.** These come from scanned
paper documents. PyMuPDF returns empty strings for such pages. Your ingestion
pipeline appears to work but produces zero chunks.

Let's simulate this: create a PDF with just an image (no text layer) and see
what happens.

In [ ]:
# Create a fake 'scanned' PDF — one page with just a filled rectangle, no text
import pymupdf

scanned_path = SAMPLE_DIR / "scanned_fake.pdf"
scan_doc = pymupdf.open()
page = scan_doc.new_page()
# Draw a filled rectangle to simulate a scanned image with no text layer
page.draw_rect(pymupdf.Rect(100, 100, 500, 700), fill=(0.9, 0.9, 0.9))
scan_doc.save(str(scanned_path))
scan_doc.close()

# Now try to extract text
doc = pymupdf.open(scanned_path)
extracted = doc[0].get_text()
doc.close()

print(f"Fake 'scanned' PDF: {scanned_path.name}")
print(f"Extracted text length: {len(extracted)} chars")
print(f"Extracted content: {extracted!r}")
print()
print("That's the trap. The parser succeeded — no crash. But zero content.")
print("Your ingestion silently produced 0 chunks from this document.")

# Cleanup — don't clutter sample_docs/
scanned_path.unlink()

**Fix (out of scope for W8):** run OCR on scanned PDFs before parsing.
Popular options: `pytesseract`, `easyocr`, or cloud services (AWS Textract,
Google Document AI). W10 introduces OCR as an optional add-on if your corpus
needs it.

**For W8 today:** add a check to your ingestion pipeline: if a PDF parses to
< 50 chars of text, log a warning. That'll catch the silent-failure case.

---

## Cell 7 — Topic 1 recap

**Four libraries, one decision matrix:**

| Format | Library | Speed | Structure | Tables |
|---|---|---|---|---|
| PDF (prose) | PyMuPDF | fast | pages only | poor |
| PDF (tables) | pdfplumber | slower | pages only | good |
| HTML | BeautifulSoup | fast | tags | table tags |
| DOCX | python-docx | fast | styles (best!) | table objects |

**Traps to remember:**
- HTML: don't include `<nav>` and `<footer>` — they leak into your chunks
- PDF: some PDFs are scanned images → empty text → silent failure
- All formats: check output length before assuming parse succeeded

Now to Topic 2 — chunking.

---

## Cell 8 — Chunking strategy #1: Fixed-size (W6's baseline)

Fixed-size chunking is the sliding window from W6. Simple, but cuts through
sentences and paragraphs indiscriminately.

Let's run it on the PDF's page 1 text and see the damage.

In [ ]:
# Load the PDF text (page 1 only, for demo compactness)
doc = pymupdf.open(SAMPLE_PDF)
page1_text = doc[0].get_text()
doc.close()

def chunk_fixed(text, size=400, overlap=40):
    """W6's sliding-window chunker."""
    if len(text) <= size:
        return [text]
    chunks, i = [], 0
    while i < len(text):
        end = min(i + size, len(text))
        chunks.append(text[i:end])
        if end == len(text):
            break
        i = end - overlap
    return chunks

fixed_chunks = chunk_fixed(page1_text, size=400, overlap=40)
print(f"Fixed-size chunking (size=400, overlap=40) → {len(fixed_chunks)} chunks\n")

for i, chunk in enumerate(fixed_chunks):
    ends_mid = not chunk.rstrip().endswith(('.', '!', '?', ':'))
    marker = " ← ends mid-sentence" if ends_mid else ""
    preview = chunk[-60:].replace('\n', ' ')
    print(f"  [{i}] {len(chunk):>4d} chars, ends: '...{preview}'{marker}")

**Discussion:** count the mid-sentence cuts. Each one is a chunk that starts
with fragment-context — bad for retrieval. This is why we need better chunking.

---

## Cell 9 — Chunking strategy #2: Recursive

**Recursive chunking:** try paragraphs first; if a paragraph is too long, fall
back to sentence splits within it. Preserves boundaries whenever possible.

This is the strategy most systems default to.

In [ ]:
import re

def chunk_recursive(text, max_size=400):
    """Split by paragraphs. If a paragraph exceeds max_size, split by sentences."""
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    for para in paragraphs:
        if len(para) <= max_size:
            chunks.append(para)
        else:
            # Sentence-level fallback
            sentences = re.split(r'(?<=[.!?])\s+', para)
            current = ""
            for sent in sentences:
                if len(current) + len(sent) + 1 <= max_size:
                    current = (current + " " + sent).strip()
                else:
                    if current:
                        chunks.append(current)
                    current = sent
            if current:
                chunks.append(current)
    return chunks

recursive_chunks = chunk_recursive(page1_text, max_size=400)
print(f"Recursive chunking (max_size=400) → {len(recursive_chunks)} chunks\n")

for i, chunk in enumerate(recursive_chunks):
    ends_mid = not chunk.rstrip().endswith(('.', '!', '?', ':'))
    marker = " ← ends mid-sentence" if ends_mid else " ← clean end"
    preview = chunk[-60:].replace('\n', ' ')
    print(f"  [{i}] {len(chunk):>4d} chars, ends: '...{preview}'{marker}")

**Compare with Cell 8:**
- Fixed had multiple mid-sentence cuts → messy chunks
- Recursive respects paragraph AND sentence boundaries → cleaner chunks
- Chunk sizes are variable (that's the trade-off), but every chunk is
  semantically self-contained

**Recursive is the sane default for most RAG systems.** Use it unless you have
a specific reason to try something else.

---

## Cell 10 — Chunking strategy #3: Semantic (simplified demo)

**Semantic chunking:** embed each sentence, group consecutive sentences whose
cosine similarity is above a threshold. Similar sentences stay together;
topic shifts trigger new chunks.

**Full semantic chunking** would embed every sentence in a long document
(expensive on API cost). **Simplified demo:** hand-pick 6 sentences covering
2 topics, embed them, show the groupings.

In [ ]:
import os
import numpy as np
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY for the semantic chunking demo"

client = OpenAI()

# 6 sentences, 2 topics — leave policy + remote work
sentences = [
    "Employees receive 20 days of paid annual leave per year.",
    "Leave requests should be submitted at least two weeks in advance.",
    "Unused leave carries over up to 5 days to the following year.",
    "All employees are eligible for hybrid work arrangements.",
    "The standard expectation is three days in-office per week.",
    "Full-remote status requires VP approval and a business case.",
]

# Embed all 6 in one API call (cheap: ~$0.0001)
resp = client.embeddings.create(
    model="text-embedding-3-small",
    input=sentences,
)
embeddings = [np.array(item.embedding) for item in resp.data]

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Show consecutive-sentence similarities
print("Consecutive sentence similarities:\n")
for i in range(len(sentences) - 1):
    sim = cosine(embeddings[i], embeddings[i+1])
    print(f"  s{i} → s{i+1}: cosine = {sim:.3f}")
    print(f"    s{i}: {sentences[i]}")
    print(f"    s{i+1}: {sentences[i+1]}\n")

In [ ]:
# Now group consecutive sentences into chunks — split whenever similarity drops below threshold
THRESHOLD = 0.55  # tuned for this demo; real systems calibrate per corpus

chunks = [[sentences[0]]]
for i in range(1, len(sentences)):
    sim = cosine(embeddings[i-1], embeddings[i])
    if sim >= THRESHOLD:
        chunks[-1].append(sentences[i])
    else:
        chunks.append([sentences[i]])  # new chunk starts here

print(f"Semantic chunking (threshold={THRESHOLD}) → {len(chunks)} chunks\n")
for i, chunk_sentences in enumerate(chunks):
    print(f"══ Chunk {i} ({len(chunk_sentences)} sentences) ══")
    for s in chunk_sentences:
        print(f"  • {s}")
    print()

**Discussion:**
- Did the chunker group the 3 leave-policy sentences together and separate them
  from the remote-work sentences? Look at the similarity between s2 and s3.
- This works when your corpus has natural topic shifts. It's overkill for
  short structured documents.
- **Cost warning:** on a large corpus you'd embed every sentence at ingestion
  time. For a 100-doc corpus that's ~2000 sentences × ~40 tokens = ~80K tokens
  = $0.002. Cheap. But at 10K docs it becomes $0.20 — still cheap.
- **When to use:** documents without clear headings but with clear topic shifts
  (blog posts, articles, transcripts).

---

## Cell 11 — Chunking strategy #4: Structure-aware

**Structure-aware chunking:** use the document's OWN structure as split points.
For DOCX, use heading styles. For HTML, use `<h1>` / `<h2>`. For Markdown, use
`#` / `##`.

This is the **best strategy when your documents have real structure** — which
the DOCX file does.

In [ ]:
from docx import Document

def chunk_docx_structure_aware(path):
    """Split a DOCX into chunks bounded by Heading 1 / Heading 2 styles.
    Attach section_path metadata based on the heading hierarchy."""
    doc = Document(str(path))
    
    chunks = []
    current_h1 = None
    current_h2 = None
    current_text = []
    
    def flush():
        if current_text:
            section_path = " > ".join(p for p in [current_h1, current_h2] if p)
            chunks.append({
                "section_path": section_path,
                "text": "\n".join(current_text),
            })
    
    for para in doc.paragraphs:
        if not para.text.strip():
            continue
        style = para.style.name
        
        if style == "Title" or style == "Heading 1":
            flush()  # save what we have
            current_text = []
            current_h1 = para.text
            current_h2 = None
        elif style == "Heading 2":
            flush()  # save subsection
            current_text = []
            current_h2 = para.text
        else:
            current_text.append(para.text)
    
    flush()  # final flush
    return chunks

structured = chunk_docx_structure_aware(SAMPLE_DOCX)
print(f"Structure-aware chunking of {SAMPLE_DOCX.name} → {len(structured)} chunks\n")

for i, chunk in enumerate(structured):
    print(f"══ Chunk {i} ══")
    print(f"  section_path: {chunk['section_path']!r}")
    preview = chunk['text'][:120].replace('\n', ' ')
    print(f"  text:         {preview}...")
    print(f"  length:       {len(chunk['text'])} chars\n")

**Look at that.** Each chunk has:
- A `section_path` like `'New Employee Onboarding Guide > IT Support Contacts'`
- Text that is exactly the content under that heading — no more, no less

**Why this is powerful:**
1. **Cleaner retrieval** — a chunk about 'IT Support Contacts' won't drift into 'Week 2' content
2. **Metadata comes for free** — `section_path` is already extracted, ready for Day 2's metadata schema
3. **Explainable citations** — 'this answer came from section: Onboarding > IT Support Contacts'

**Rule of thumb:** if your document has headings, use structure-aware chunking.
It's the biggest single quality lift you can make to a RAG pipeline.

---

## Cell 12 — Wrap + Homework for Day 2

**Today you:**
1. Parsed 3 real document formats — PyMuPDF, pdfplumber, BeautifulSoup, python-docx
2. Saw the scanned-PDF trap (empty text → silent failure)
3. Compared 4 chunking strategies on real content — fixed, recursive, semantic, structure-aware
4. Confirmed structure-aware is the biggest quality lift when documents have headings

**Decision matrix for your capstone (in Track B):**

| Your corpus looks like… | Chunking strategy |
|---|---|
| Docs with clear headings (policies, manuals, guides) | Structure-aware |
| Narrative text with topic shifts (blog posts, articles) | Semantic |
| Mixed prose without structure | Recursive |
| Very short / uniform docs | Fixed |

**Day 2 will build on this:**
- Metadata enrichment — the 9-field schema, plus Qdrant filter DSL
- PII detection + redaction — regex baseline + Presidio + audit
- Wire it all together in a mini-pipeline

---

## HOMEWORK BEFORE DAY 2 — install Presidio (~5 min)

**Do this BEFORE Day 2 starts. Downloads about 150 MB. Not something to do**
**in class.**

```bash
pip install presidio-analyzer presidio-anonymizer
python -m spacy download en_core_web_sm
```

**Verify with:**
```python
from presidio_analyzer import AnalyzerEngine
engine = AnalyzerEngine()
print("Presidio ready.")
```

If this fails on Vocareum, tell your instructor before Day 2 — you'll need
help resolving Docker or environment issues that can't be sorted in class.